## Setup — kiểm tra môi trường

In [40]:
from importlib.metadata import version
import importlib
import tiktoken
import torch
print("PyTorch version:", torch.__version__)


print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

PyTorch version: 2.13.0+cu129
torch version: 2.13.0+cu129
tiktoken version: 0.13.0


## 2.2 Tokenizing text (trang 26–31)

In [33]:
with open('the-verdict.txt', 'r', encoding='utf-8') as f:
  raw_text = f.read()
  
print("Total number of chracter:", len(raw_text))
print(raw_text[:99])

Total number of chracter: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [34]:
import re
text = "Hello, world. This, is a test."
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [35]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 2.3 Converting tokens into token IDs (trang 31–37)

Đề bài chi tiết: xem `REQUIREMENTS.md`.

### Task 1 — Xây dựng vocabulary

In [36]:
vocab = sorted(set(preprocessed))

kv = dict()
for k, item in enumerate(vocab):
  kv[item] = k
print(kv)

vocab = {item:k for k, item in enumerate(vocab)}
print(vocab)

{'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29, 'Devonshire': 30, 'Don': 31, 'Dubarry': 32, 'Emperors': 33, 'Florence': 34, 'For': 35, 'Gallery': 36, 'Gideon': 37, 'Gisburn': 38, 'Gisburns': 39, 'Grafton': 40, 'Greek': 41, 'Grindle': 42, 'Grindles': 43, 'HAD': 44, 'Had': 45, 'Hang': 46, 'Has': 47, 'He': 48, 'Her': 49, 'Hermia': 50, 'His': 51, 'How': 52, 'I': 53, 'If': 54, 'In': 55, 'It': 56, 'Jack': 57, 'Jove': 58, 'Just': 59, 'Lord': 60, 'Made': 61, 'Miss': 62, 'Money': 63, 'Monte': 64, 'Moon-dancers': 65, 'Mr': 66, 'Mrs': 67, 'My': 68, 'Never': 69, 'No': 70, 'Now': 71, 'Nutley': 72, 'Of': 73, 'Oh': 74, 'On': 75, 'Once': 76, 'Only': 77, 'Or': 78, 'Perhaps': 79, 'Poor': 80, 'Professional': 81, 'Renaissance': 82, 'Ri

### Task 2 — Cài đặt class `SimpleTokenizerV1`

In [37]:
class SimpleTokenizerV1:
  def __init__(self, vocab: dict[str, int]):
    self.str_to_int: dict[str, int] = vocab
    self.int_to_str: dict[int, str] = { item:num for num, item in vocab.items()}

  def encode(self, text: str) -> list[int]:
    preprocessed: list[str] = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    ids: list[int] = [self.str_to_int[s] for s in preprocessed]
    return ids
    
  def decode(self, ids: list[int]) -> list[str]:
    result = " ".join(self.int_to_str[id] for id in ids)
    result = re.sub(r'\s+([,.?!"()\'])', r'\1', result)
    return result
      
print("vocab: ", vocab)
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable
pride."""
ids = tokenizer.encode(text)
tokenizer.decode(ids)
    

vocab:  {'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29, 'Devonshire': 30, 'Don': 31, 'Dubarry': 32, 'Emperors': 33, 'Florence': 34, 'For': 35, 'Gallery': 36, 'Gideon': 37, 'Gisburn': 38, 'Gisburns': 39, 'Grafton': 40, 'Greek': 41, 'Grindle': 42, 'Grindles': 43, 'HAD': 44, 'Had': 45, 'Hang': 46, 'Has': 47, 'He': 48, 'Her': 49, 'Hermia': 50, 'His': 51, 'How': 52, 'I': 53, 'If': 54, 'In': 55, 'It': 56, 'Jack': 57, 'Jove': 58, 'Just': 59, 'Lord': 60, 'Made': 61, 'Miss': 62, 'Money': 63, 'Monte': 64, 'Moon-dancers': 65, 'Mr': 66, 'Mrs': 67, 'My': 68, 'Never': 69, 'No': 70, 'Now': 71, 'Nutley': 72, 'Of': 73, 'Oh': 74, 'On': 75, 'Once': 76, 'Only': 77, 'Or': 78, 'Perhaps': 79, 'Poor': 80, 'Professional': 81, 'Renaissance':

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

### Task 3 — Kiểm tra giới hạn của tokenizer

In [38]:
text = "Hello, do you like tea"
print(tokenizer.encode(text))

KeyError: 'Hello'

## 2.4 Adding special context tokens (trang 37–42)

Đề bài chi tiết: xem `REQUIREMENTS.md`.

### Task 1 — Mở rộng vocabulary với token đặc biệt

In [ ]:

all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}
print(vocab['<|unk|>'])
for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)

1131
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


### Task 2 — Cài đặt class `SimpleTokenizerV2`

In [ ]:
class SimpleTokenizerV2:
  def __init__(self, vocab: dict[str, int]):
    self.str_to_int: dict[str, int] = vocab
    self.int_to_str: dict[int, str] = { item:num for num, item in vocab.items()}

  def encode(self, text: str) -> list[int]:
    preprocessed: list[str] = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    
    ids: list[int] = [self.str_to_int[s] if s in self.str_to_int else self.str_to_int["<|unk|>"] for s in preprocessed]

    return ids
    
  def decode(self, ids: list[int]) -> list[str]:
      tokens = []
      for id in ids:
        token = self.int_to_str[id]
        tokens.append(token)
      result = " ".join(tokens)
      result = re.sub(r'\s+([,.?!"()\'])', r'\1', result)
      return result

tokenizer = SimpleTokenizerV2(vocab)
text1 = "Hello, do you sdfds like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
ids = tokenizer.encode(text)
print(tokenizer.decode(ids))

<|unk|>, do you <|unk|> like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## 2.5 Byte pair encoding (trang 42–44)

Đề bài chi tiết: xem `REQUIREMENTS.md`.

### Task 1 — Dùng tokenizer BPE của `tiktoken`

In [ ]:
import importlib
import tiktoken

print("tiktoken version: ", importlib.metadata.version("tiktoken"))
tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

strings = tokenizer.decode(integers)
print(strings)

tiktoken version:  0.13.0
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


## 2.6 Data sampling with a sliding window (trang 45–53)

Đề bài chi tiết: xem `REQUIREMENTS.md`.

### Task 1 — Sliding window cơ bản (x/y pairs)

In [ ]:
with open('./the-verdict.txt', 'r', encoding='utf-8') as f:
  raw_text = f.read()
  
enc_text = tokenizer.encode(raw_text)
context_size = 4
# dec_text = tokenizer.decode(enc_text)
# print("raw_text: ", raw_text[:50])
# print("enc_text: ", enc_text[:50])

# for num, value in enumerate(enc_text):
#   x = enc_text[num : num + context_size]
#   y = enc_text[num+1:num+1 + context_size]

#   print(f"{tokenizer.decode(x)} ----> {tokenizer.decode([enc_text[num+context_size]])} " )
#   print(y)

enc_sample = enc_text[40:]
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 had ---->  dropped
 had dropped ---->  his
 had dropped his ---->  painting
 had dropped his painting ----> ,


### Task 2 — Cài đặt `GPTDatasetV1` + `create_dataloader_v1`

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

### Task 3 — Test dataloader

In [43]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
dataloader = create_dataloader_v1(
    raw_text, batch_size=4, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619],
        [1464, 1807, 3619,  402]])

Targets:
 tensor([[ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619],
        [1464, 1807, 3619,  402],
        [1807, 3619,  402,  271]])


# 2.7 Creating token embeddings

In [57]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

print(embedding_layer(torch.tensor([3])))

print(embedding_layer(input_ids))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


# 2.8 Encoding word positions

In [82]:
vocab_size = 50257
output_dim = 3
torch.manual_seed(11)
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(token_embedding_layer.weight)
print("tensor 40: ", token_embedding_layer(torch.tensor([40])))
max_length = 4
dataloader = create_dataloader_v1(
  raw_text, batch_size=8, max_length=max_length,
  stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs: \n", inputs)
print("\nInputs shape: \n", inputs.shape)

token_embeddings = token_embedding_layer(inputs)
print("token embedding shape: ", token_embeddings.shape)

# positoin embeddings
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print("pos_embeddings shape: ", pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)
print(input_embeddings)

Parameter containing:
tensor([[-0.5108,  1.0283, -0.3532],
        [ 0.1230, -0.1816, -1.4972],
        [ 0.1421, -0.5243, -0.2487],
        ...,
        [ 1.1795,  1.2332, -2.1494],
        [ 0.2964,  0.6474,  0.3677],
        [-1.8952, -0.7238, -0.2536]], requires_grad=True)
tensor 40:  tensor([[ 0.3890, -2.2679, -0.6950]], grad_fn=<EmbeddingBackward0>)
Token IDs: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape: 
 torch.Size([8, 4])
token embedding shape:  torch.Size([8, 4, 3])
pos_embeddings shape:  torch.Size([4, 3])
torch.Size([8, 4, 3])
tensor([[[-6.5253e-02, -1.6378e+00, -1.1908e+00],
         [ 6.1559e-02,  2.6371e-01,  6.6869e-01],
         [ 1.1739e+00,  9.2364e-01, -1.8967e+00],
         [ 1.6205e+00,  5.90